In [11]:
import pandas as pd
import numpy as np
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer

pd.set_option('display.max_rows', 50)

df_test = pd.read_csv('test_data.csv')
df_test.head()
df_test['ID 1'].value_counts()

ID 1
St 48          212
St 37           23
StSi             9
St 48-Th         6
St 48-BK?        6
St 48Th-BK?      2
St48-BK?         2
St37-BK?         1
St 48-BK         1
Name: count, dtype: int64

In [2]:
# Auswählen der Spalten, die skaliert werden sollen
columns_to_scale = ['Fe', 'C', 'Si', 'Mn', 'P', 'S', 'Cr', 'Mo', 'Ni', 'Al', 'Co', 'Cu', 'Nb', 'Ti', 'V', 'W', 'Pb', 'Sn', 'B', 'Ca', 'Zr', 'As', 'Bi']

# Entferne Zeilen, bei denen in 'ID 1' ein '?' vorkommt
# df_test = df_test.loc[~df_test['ID 1'].str.contains('\?', na=False)]

# Skalieren der ausgewählten Spalten
scaler = FunctionTransformer(np.log1p, validate=True)
scaled_data = scaler.fit_transform(df_test[columns_to_scale])

# Erstellen eines neuen DataFrame für die skalierten Daten
df_scaled = pd.DataFrame(scaled_data, columns=columns_to_scale)
df_scaled.head()

,Fe,C,Si,Mn,P,S,Cr,Mo,Ni,Al,...,Ti,V,W,Pb,Sn,B,Ca,Zr,As,Bi
0,0.0,0.032564,0.185649,0.531216,0.028685,0.029073,0.007472,0.005087,0.033725,0.001998,...,0.001000,0.001998,0.039221,0.014889,0.007968,0.001,0.001000,0.001998,0.049457,0.011336
1,0.0,0.150143,0.122218,0.542324,0.021468,0.040278,0.005187,0.004092,0.038259,0.017152,...,0.003195,0.001798,0.039221,0.034112,0.012126,0.001,0.001299,0.001998,0.061001,0.010742
2,0.0,0.134531,0.235072,0.470004,0.018429,0.025765,0.004988,0.004888,0.043730,0.002497,...,0.001199,0.001798,0.039221,0.014889,0.018920,0.001,0.001000,0.001998,0.061847,0.009950
3,0.0,0.234281,0.242162,0.486738,0.012818,0.030723,0.004988,0.004888,0.038836,0.002297,...,0.001199,0.002297,0.039221,0.014889,0.008861,0.001,0.001000,0.001998,0.034691,0.009950
4,0.0,0.227136,0.253091,0.496524,0.011336,0.056097,0.004988,0.003992,0.026934,0.002297,...,0.001399,0.002297,0.039221,0.014889,0.010544,0.001,0.001000,0.001998,0.043443,0.010247


In [ ]:
#X_train, X_test = train_test_split(df_scaled,test_size=0.2, random_state=42)

# cntr: Cluster centers
# u: Final fuzzy partition matrix
# u0: Initial guess at fuzzy partition matrix
# d: Final Euclidean distance matrix
# jm: Objective function history
# p: Number of iterations run
# fpc: Fuzzy partition coefficient
#cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(X_train.transpose(),3,1.4, error=0.01, maxiter=10000, init=None)

#Cluster Centers (cntr): This shows the center of each cluster. 
#You can compare these centers to understand how different clusters differ from each other in terms of chemical properties.
#print(cntr)
#Membership Matrix (u): Each element's degree of belonging to each cluster. 
#A higher value indicates a stronger association with that cluster.
#print(u)
#Fuzzy Partition Coefficient (fpc): A value close to 1 indicates well-separated clusters. 
#Lower values indicate overlapping clusters.
#print(fpc)

#u, u0, d, jm, p, fpc = fuzz.cluster.cmeans_predict(df.transpose(), cntr, 5, error=0.01, maxiter=10000)

#print(fpc)

In [5]:
def fuzzy_c_means_cluster(data, labels_df, name_of_label, number_of_cluster):
    
    # Standardizing the Data
    scaler = FunctionTransformer(np.log1p, validate=True)
    data_scaled = scaler.fit_transform(data)
    
    # Apply fuzzy-c-mean with hard parameters
    cntr, u1, u0, d, jm, p, fpc = fuzz.cluster.cmeans(data_scaled.transpose(), number_of_cluster, 1.5, error=0.005, maxiter=10000, init=None)
    
    # Transpose fuzzy matrix u
    u_df = pd.DataFrame(u1.T)
    
    # Create cluster membership overview
    cluster_df = pd.DataFrame(u_df.values.round(6), columns=[f'cluster {i+1}' for i in range(u_df.shape[1])], index=data.index)
    
    # Assign each data point to the cluster with the highest membership
    max_membership_indices = np.argmax(u1, axis=0)

    # Count the number of data points in each cluster
    cluster_counts = np.bincount(max_membership_indices, minlength=u1.shape[0])

    # Add the cluster_counts as the top row of cluster_df
    cluster_df.loc['Insgesamt'] = cluster_counts

    # Add labels to the cluster_df
    cluster_df[name_of_label] = labels_df[name_of_label]

    print("Der folgende Wert gibt an wie klar die Cluster getrennt sind, umso näher an 1 desto klarer sind sie getrennt:", fpc.round(5))
    
    return cluster_df

In [6]:
fuzzy_c_means_cluster(df_scaled,df_test,'ID 1',9)

Der folgende Wert gibt an wie klar die Cluster getrennt sind, umso näher an 1 desto klarer sind sie getrennt: 0.58601


,cluster 1,cluster 2,cluster 3,cluster 4,cluster 5,cluster 6,cluster 7,cluster 8,cluster 9,ID 1
0,0.118956,0.277263,0.124360,0.075118,0.036265,0.007219,0.239027,0.065460,0.056332,St 48
1,0.037102,0.659201,0.114644,0.055221,0.031444,0.000473,0.052082,0.040275,0.009556,St 48
2,0.113161,0.046350,0.070006,0.027211,0.002539,0.000648,0.711845,0.026158,0.002082,St 48
3,0.756183,0.003373,0.107447,0.059582,0.000401,0.000081,0.023227,0.049553,0.000151,St 48
4,0.824045,0.003675,0.038235,0.097694,0.000501,0.000133,0.019027,0.016507,0.000183,St 48
...,...,...,...,...,...,...,...,...,...,...
258,0.008424,0.092957,0.019257,0.015414,0.612661,0.000459,0.019954,0.013083,0.217790,St 48-BK?
259,0.006782,0.084144,0.014778,0.011416,0.315430,0.000402,0.017645,0.009712,0.539690,St37-BK?
260,0.006782,0.084144,0.014778,0.011416,0.315430,0.000402,0.017645,0.009712,0.539690,St48-BK?
261,0.013220,0.101817,0.035632,0.025495,0.698111,0.000621,0.024073,0.027752,0.073279,St 48-BK


In [7]:
df_sv = pd.read_csv('labeled_OES_data_proper.csv')

# Auswählen der Spalten, die skaliert werden sollen
columns_to_scale = ['C','Si','Mn','P','S','Cr','Ni','Al','Cu']

# Skalieren der ausgewählten Spalten
scaler = FunctionTransformer(np.log1p, validate=True)
scaled_data = scaler.fit_transform(df_sv[columns_to_scale])

# Erstellen eines neuen DataFrame für die skalierten Daten
df_scaled1 = pd.DataFrame(scaled_data, columns=columns_to_scale)
df_scaled1

,C,Si,Mn,P,S,Cr,Ni,Al,Cu
0,0.254384,0.186341,0.425377,0.024579,0.043618,0.034321,0.048203,0.002081,0.134822
1,0.092837,0.680568,0.515614,0.026090,0.034031,0.009141,0.041702,0.003128,0.180653
2,0.124575,0.606590,0.641678,0.014741,0.046470,0.023196,0.106310,0.053288,0.311643
3,0.118079,0.583425,0.672604,0.014839,0.045531,0.029510,0.106760,0.049044,0.312619
4,0.211187,0.213959,0.547626,0.027309,0.041224,0.014776,0.048668,0.004205,0.154069
...,...,...,...,...,...,...,...,...,...
432,0.153193,0.174163,0.210463,0.116938,0.008481,0.003394,0.025034,0.010363,0.031870
433,0.165345,0.093873,0.233838,0.082520,0.061396,0.007015,0.014731,0.000860,0.047914
434,0.004540,0.121377,0.065694,0.122218,0.049266,0.003344,0.025790,0.005137,0.054346
435,0.061048,0.077979,0.157538,0.122218,0.024644,0.002846,0.048076,0.004440,0.135448


In [8]:
fuzzy_c_means_cluster(df_scaled1,df_sv,'steel_class',5)

Der folgende Wert gibt an wie klar die Cluster getrennt sind, umso näher an 1 desto klarer sind sie getrennt: 0.76776


,cluster 1,cluster 2,cluster 3,cluster 4,cluster 5,steel_class
0,0.746658,0.004460,0.000765,0.001927,0.246190,St 48
1,0.001223,0.000196,0.997354,0.000268,0.000959,St Si
2,0.026629,0.004385,0.939034,0.004189,0.025763,St Si
3,0.033182,0.005571,0.922436,0.005034,0.033776,St Si
4,0.002831,0.000027,0.000014,0.000010,0.997117,St 48
...,...,...,...,...,...,...
433,0.055464,0.166488,0.002615,0.745828,0.029605,Puddelstahl
434,0.013728,0.035174,0.002821,0.940028,0.008250,Puddelstahl
435,0.012805,0.055850,0.001307,0.922738,0.007300,Puddelstahl
436,0.012997,0.030177,0.002639,0.946549,0.007638,Puddelstahl


In [9]:
def fuzzy_c_means_clustering(data):
    # Extracting features and class labels
    features = data.iloc[:, :-1].values.T
    labels = data.iloc[:, -1]

    # Perform fuzzy c-means clustering
    cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
        features, 5, 1.1, error=0.005, maxiter=1000, init=None
    )

    # Determining the cluster to which each data point belongs
    cluster_membership = np.argmax(u, axis=0)

    # Creating a dataframe to count occurrences
    cluster_counts = pd.DataFrame(0, index=np.unique(labels), columns=np.arange(5))

    # Counting occurrences
    for label, cluster in zip(labels, cluster_membership):
        cluster_counts.loc[label, cluster] += 1

    return cluster_counts

In [10]:
fuzzy_c_means_clustering(df_sv)

,0,1,2,3,4
Flussstahl,0,9,0,0,0
Puddelstahl,0,22,0,4,0
St 37,2,91,0,4,0
St 48,145,7,10,111,1
St Si,0,0,0,0,31
